In [ ]:
import pandas as pd
import os
import matplotlib.pyplot as plt

In [ ]:
csv_files = []
for filename in os.listdir("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive"):
    if filename.endswith('.csv'):
        csv_files.append(os.path.join("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predicts\\archive", filename))


csv_files

In [ ]:
circuits = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\circuits.csv")
drivers = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\drivers.csv")
constructor_results = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\constructor_results.csv")
constructor_standings = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\constructor_standings.csv")
constructors = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\constructors.csv")
lap_times = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\lap_times.csv")
pit_stops = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\pit_stops.csv")
qualifying = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\qualifying.csv")
races = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\races.csv")
results = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\results.csv")
status = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\status.csv") 
sprint_results = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\sprint_results.csv")
driver_standings = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\driver_standings.csv")
seasons = pd.read_csv("C:\\Users\\Miguel António\\Desktop\\PORTFOLIO\\f1_predictions\\archive\\seasons.csv")

In [ ]:
dataframes = {
    "circuits": circuits,
    "drivers": drivers,
    "driver_standings": driver_standings,
    "constructor_results": constructor_results,
    "constructor_standings": constructor_standings,
    "constructors": constructors,
    "lap_times": lap_times,
    "pit_stops": pit_stops,
    "qualifying": qualifying,
    "races": races,
    "results": results,
    "status": status,
    "sprint_results": sprint_results,
    "seasons": seasons}

In [ ]:
for elem in dataframes:
    print(f"{elem} has {dataframes[elem].shape[0]} rows and {dataframes[elem].shape[1]} columns {dataframes[elem].nunique()} unique values")
    print("\n")

In [ ]:
# Verifying the NaN values in the DataFrame
for name, df in dataframes.items():
    nan_count = df.isna().sum().sum()
    if nan_count > 0:
        print(f"{name} has {nan_count} NaN values.")
    else:
        print(f"{name} has no NaN values.")

In [ ]:
qualidfying_nan = qualifying.isna().sum()
print(qualidfying_nan)
print("\nqualifying has", qualifying.shape[0], "rows and", qualifying.shape[1], "columns.")

In [ ]:
qualifying = qualifying.dropna(subset=['q1', 'q2', 'q3'])
print("After dropping NaN values in qualifying, the DataFrame has:")    
print(qualifying.shape[0], "rows and", qualifying.shape[1], "columns.")

In [ ]:
# Check for duplicates 
for name, df in dataframes.items():
    duplicates = df.duplicated().sum()
    if duplicates > 0:
        print(f"{name} has {duplicates} duplicate rows.")
    else:
        print(f"{name} has no duplicate rows.")


In [ ]:
# Check for catehgorical variables
for name, df in dataframes.items():
    categorical_cols = df.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        print(f"{name} has the following categorical columns: {categorical_cols.tolist()}")
    else:
        print(f"{name} has no categorical columns.")


print("\n")
# Check for numerical variables
for name, df in dataframes.items():
    numerical_cols = df.select_dtypes(include=['number']).columns
    if len(numerical_cols) > 0:
        print(f"{name} has the following numerical columns: {numerical_cols.tolist()}")
    else:
        print(f"{name} has no numerical columns.")
    

In [ ]:
def time_str_to_millis(time_str):
    try:
        # Split "1:26.572" into ["1", "26.572"]
        minutes, sec_millis = time_str.split(':')
        seconds, millis = sec_millis.split('.')
        
        total_millis = (int(minutes) * 60 * 1000) + (int(seconds) * 1000) + int(millis)
        return total_millis
    except:
        return pd.NA  

In [ ]:
qualifying['q1'] = qualifying['q1'].apply(time_str_to_millis)
qualifying['q2'] = qualifying['q2'].apply(time_str_to_millis)
qualifying['q3'] = qualifying['q3'].apply(time_str_to_millis)   

In [ ]:
results['fastestLapTime'] = results['fastestLapTime'].apply(time_str_to_millis)

In [ ]:
sprint_results['fastestLapTime'] = sprint_results['fastestLapTime'].apply(time_str_to_millis)

In [ ]:
# Check which dataframes have the columns fp1_time, fp2_time, fp3_time, qualifying_time, sprint_time and fp1_date, fp2_date, fp3_date
for name, df in dataframes.items():
    if 'fp1_time' in df.columns or 'fp2_time' in df.columns or 'fp3_time' in df.columns or 'qualifying_time' in df.columns or 'sprint_time' in df.columns or 'fp1_date' in df.columns or 'fp2_date' in df.columns or 'fp3_date' in df.columns:
        print(f"{name} has the columns fp1_time, fp2_time, fp3_time, qualifying_time, sprint_time and fp1_date, fp2_date, fp3_date.")
    else:
        print(f"{name} does not have the columns fp1_time, fp2_time, fp3_time, qualifying_time, sprint_time and fp1_date, fp2_date, fp3_date.")

In [ ]:
# Create the features calendar interval and momento of the day on the races DataFrame
def create_calendar_interval_and_momento_of_day(df):
    df['day'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    df['moment_of_day'] = pd.to_datetime(df['date']).dt.hour.apply(lambda x: 'morning' if x < 12 else 'afternoon' if x < 18 else 'night')
    return df

create_calendar_interval_and_momento_of_day(races)

In [ ]:
# Comvert dob (date of birth) to datetime
drivers['dob'] = pd.to_datetime(drivers['dob'], errors='coerce')

In [ ]:
# Create a start/middle/end division of the calendar based on the month of the day variable
def create_calendar_division(df):
    df['calendar_division_s/m/e'] = pd.to_datetime(df['day']).dt.month.apply(lambda x: 'start' if x <= 4 else 'middle' if x <= 8 else 'end')
    return df


create_calendar_division(races)

In [ ]:
# Merge the circuits DataFrame with Races DataFrame 
circuits_races = pd.merge(races, circuits, how='inner', left_on='circuitId', right_on='circuitId')

In [ ]:
continent_map = {
    # Oceania
    'Australia': 'Oceania',
    
    # Asia
    'Malaysia': 'Asia',
    'China': 'Asia',
    'Bahrain': 'Asia',
    'Singapore': 'Asia',
    'Japan': 'Asia',
    'UAE': 'Asia',
    'Korea': 'Asia',
    'India': 'Asia',
    'Russia': 'Asia',
    'Azerbaijan': 'Asia',
    'Qatar': 'Asia',
    'Saudi Arabia': 'Asia',

    # Europe
    'Spain': 'Europe',
    'Monaco': 'Europe',
    'Turkey': 'Europe',
    'UK': 'Europe',
    'Germany': 'Europe',
    'Hungary': 'Europe',
    'Belgium': 'Europe',
    'Italy': 'Europe',
    'France': 'Europe',
    'Austria': 'Europe',
    'Portugal': 'Europe',
    'Netherlands': 'Europe',
    'Sweden': 'Europe',
    'Switzerland': 'Europe',

    # America
    'Brazil': 'America',
    'Canada': 'America',
    'USA': 'America',
    'United States': 'America',
    'Argentina': 'America',
    'Mexico': 'America',

    # Africa
    'South Africa': 'Africa',
    'Morocco': 'Africa',
}

In [ ]:
# Create a calendar division based on the name variable (Europe, Asia, America, Oceania)
def create_calendar_region(df):
    df['country'] = df['country'].replace({'USA': 'United States'})

    df['calendar_region'] = df['country'].map(continent_map)
    return df


create_calendar_region(circuits_races)

In [ ]:
unique_circuits = circuits_races.drop_duplicates(subset='circuitId')

continent_counts = unique_circuits['calendar_region'].value_counts()

plt.figure(figsize=(10, 6))
continent_counts.plot(kind='bar', color='mediumseagreen')
plt.title('Número de Circuitos Únicos por Continente')
plt.xlabel('Continente')
plt.ylabel('Número de Circuitos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Check the columns that are categorical variables
for name, df in dataframes.items():
    categorical_cols = df.select_dtypes(include=['object']).columns
    if len(categorical_cols) > 0:
        print(f"{name} has the following categorical columns: {categorical_cols.tolist()}")
    else:
        print(f"{name} has no categorical columns.")

        

In [ ]:
# Check the categorical variables on the circuits_races DataFrame
categorical_cols = circuits_races.select_dtypes(include=['object']).columns
if len(categorical_cols) > 0:
    print(f"circuits_races has the following categorical columns: {categorical_cols.tolist()}")
else:
    print(f"circuits_races has no categorical columns.")

In [ ]:
categorical_mappings = {
    'circuits': ['circuitRef', 'name', 'location', 'country', 'url'],
    'drivers': ['driverRef', 'number', 'code', 'forename', 'surname', 'nationality', 'url'],
    'driver_standings': ['positionText'],
    'constructor_results': ['status'],
    'constructor_standings': ['positionText'],
    'constructors': ['constructorRef', 'name', 'nationality', 'url'],
    'lap_times': ['time'],
    'pit_stops': ['time', 'duration'],
    'qualifying': ['q1', 'q2', 'q3'],
    'races': ['name', 'date', 'time', 'url', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time',
              'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time',
              'day', 'moment_of_day', 'calendar_division_s/m/e'],
    'results': ['number', 'position', 'positionText', 'time', 'milliseconds', 'fastestLap',
                'rank', 'fastestLapTime', 'fastestLapSpeed'],
    'status': ['status'],
    'sprint_results': ['position', 'positionText', 'time', 'milliseconds', 'fastestLap', 'fastestLapTime'],
    'seasons': ['url'],
    'circuits_races': ['name_x', 'date', 'time', 'url_x', 'fp1_date', 'fp1_time', 'fp2_date', 'fp2_time', 'fp3_date', 'fp3_time', 'quali_date', 'quali_time', 'sprint_date', 'sprint_time', 'day', 'moment_of_day', 'calendar_division_s/m/e', 'calendar_region', 'circuitRef', 'name_y', 'location', 'country', 'url_y']
}

def convert_categorical_columns(df, categorical_cols):
    for col in categorical_cols:
        df[col] = df[col].astype('category')
    return df

for df_name, cat_cols in categorical_mappings.items():
    globals()[df_name] = convert_categorical_columns(globals()[df_name], cat_cols)

In [ ]:
# Check the columns of each DataFrame and saved them in a dictionary
columns_dict = {name: df.columns.tolist() for name, df in dataframes.items()}

# Show all the columns
pd.set_option('display.max_columns', None)  
for name, columns in columns_dict.items():
    print(f"{name} columns: {columns}")
    print("\n")